# Typing, partner combinations and stage deviations

Reproduce original slot-based analysis. Then separate type presence, partner composition and exact swapped pairs. Associations describe BST, not battle usefulness or causal design effects.

In [1]:
# Resolve project root before package imports. Works from root or notebooks/.
import os
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file() and (p / "pokeml").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch this notebook inside the ml_pokemon project")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if sys.version_info[:2] != (3, 14):
    raise RuntimeError("Select project Python 3.14.8 kernel (.pokemon)")
print(f"Project: {PROJECT_ROOT} | Python: {sys.version.split()[0]} | Kernel: {sys.executable}")


Project: /Users/pano/Library/Mobile Documents/com~apple~CloudDocs/coding/portfolio/ml_pokemon | Python: 3.14.8 | Kernel: /Users/pano/venv/pokemon/.pokemon/bin/python


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from pokeml.eda.snapshot import load_analysis_snapshot

df = load_analysis_snapshot()
OUTPUT = Path("artifacts/eda/03_typing_analysis")
OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Validated analysis snapshot: {len(df):,} rows")

Validated analysis snapshot: 1,082 rows


## Original stage-relative deviations

Use original selected-form population for reproduction. Monotype, primary slot and secondary slot are separate constructions. Count labels show support; faded bars have fewer than ten records.

In [3]:
from pokeml.eda.types import extract_type_deviations, compare_type_ordering, fit_dual_type_dev_map, fit_mono_type_dev_map, type_presence_summary, type_pair_summary, compare_exact_type_ordering, bug_pair_analysis
from pokeml.eda.stages import STAGE_ORDER
from pokeml.eda.power_creep import default_species
from pokeml.visualisation.viz_eda.bst_vs_type_plot import type_deviations_plot
from pokeml.visualisation.viz_eda.bst_vs_type_order_plot import type_order_deviation_plot
groups, regular_baseline, legendary_baseline = extract_type_deviations(df)
display(groups.round(2))
groups.to_csv(OUTPUT / "original_type_deviations.csv", index=False)
fig, axes = type_deviations_plot(STAGE_ORDER, rarity="regular", baseline=regular_baseline, df=groups, plot_path=OUTPUT)
plt.close(fig)

,rarity,stage,type,construction,count,mean,median,deviation
0,regular,s1c2,bug,mono,8,252.2,247.0,-61.0
1,regular,s1c3,bug,mono,5,214.0,195.0,-105.0
2,regular,s2c2,bug,mono,3,427.7,404.0,-83.0
3,regular,s2c3,bug,mono,4,207.0,205.0,-200.0
4,regular,single,bug,mono,3,453.3,430.0,-48.0
...,...,...,...,...,...,...,...,...
347,regular,s2c2,water,dual_t2,6,487.5,495.0,8.0
348,regular,s2c3,water,dual_t2,1,410.0,410.0,5.0
349,regular,s3c3,water,dual_t2,1,530.0,530.0,7.0
350,legendary,single,water,dual_t2,1,600.0,600.0,10.0


/Users/pano/Library/Mobile Documents/com~apple~CloudDocs/coding/portfolio/ml_pokemon/pokeml/visualisation/viz_eda/bst_vs_type_plot.py:194: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Original primary/secondary comparison

Different partner mixes can produce slot differences. Diagonal means equal pooled slot deviations, not absence of causal ordering effect. Original minimum-three threshold is retained as exploratory. Legendary panel appears only when requested and uses exact pooled raw medians.

In [4]:
fig, axes = type_order_deviation_plot(STAGE_ORDER, [regular_baseline, legendary_baseline], groups, min_count=3, rarity_list=("regular", "legendary"), plot_path=OUTPUT)
plt.close(fig)
display(compare_type_ordering("s3c3", regular_baseline, groups, min_count=3).round(2))

/Users/pano/Library/Mobile Documents/com~apple~CloudDocs/coding/portfolio/ml_pokemon/pokeml/visualisation/viz_eda/bst_vs_type_order_plot.py:150: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,type_2,dual_t1_dev,dual_t2_dev,n_t1,n_t2,dev_z_t1,dev_z_t2,difference,abs_diff,baseline
2,dragon,77.0,17.0,6,5,2.60,0.45,2.15,2.15,523.0
12,normal,-38.0,-3.0,5,3,-1.52,-0.26,-1.25,1.25,523.0
16,steel,42.0,7.0,4,5,1.35,0.10,1.25,1.25,523.0
13,poison,-18.0,-33.0,3,8,-0.80,-1.34,0.54,0.54,523.0
10,ground,4.5,-6.0,4,8,0.01,-0.37,0.38,0.38,523.0
1,dark,12.0,7.0,4,7,0.27,0.10,0.18,0.18,523.0
14,psychic,-5.0,-3.0,3,5,-0.33,-0.26,-0.07,0.07,523.0


## Presence and partner combinations

Primary new comparisons use default species to reduce selected-form weighting. A Pokémon contributes once per type; family counts reveal dependence. Unordered pairs describe partners without conflating slot order.

In [5]:
species = default_species(df)
presence = type_presence_summary(species, dual_only=True)
pairs = type_pair_summary(species)
display(presence[presence.rarity.eq("regular")].round(2))
display(pairs[pairs.rarity.eq("regular")].sort_values(["stage", "median_residual"]).round(2))
presence.to_csv(OUTPUT / "dual_type_presence.csv", index=False)
pairs.to_csv(OUTPUT / "unordered_type_pairs.csv", index=False)

,rarity,stage,type,count,families,median_bst,median_residual,sparse
23,regular,s1c2,bug,19,19,304.0,-4.0,False
24,regular,s1c2,dark,8,8,339.0,31.0,True
25,regular,s1c2,dragon,3,3,362.0,54.0,True
26,regular,s1c2,electric,5,5,289.0,-19.0,True
27,regular,s1c2,fairy,3,3,285.0,-23.0,True
...,...,...,...,...,...,...,...,...
125,regular,single,poison,3,3,570.0,92.0,True
126,regular,single,psychic,8,8,476.5,-1.5,True
127,regular,single,rock,11,11,500.0,22.0,False
128,regular,single,steel,9,9,484.0,6.0,True


,rarity,stage,pair,count,families,median_bst,median_residual,sparse
55,regular,s1c2,bug / ice,1,1,185.0,-123.0,True
71,regular,s1c2,electric / poison,1,1,242.0,-66.0,True
66,regular,s1c2,dragon / flying,1,1,245.0,-63.0,True
98,regular,s1c2,ground / water,2,2,249.0,-59.0,True
54,regular,s1c2,bug / ground,1,1,266.0,-42.0,True
...,...,...,...,...,...,...,...,...
355,regular,single,ground / steel,1,1,570.0,92.0,True
358,regular,single,poison / rock,1,1,570.0,92.0,True
362,regular,single,rock / steel,1,1,570.0,92.0,True
319,regular,single,dark / dragon,2,2,580.0,102.0,True


## Does Bug pairing consistently imply lower BST?

Compare dual Bug against other dual types within stage. Then inspect exact partners and generation. Low BST does not establish worst competitive type; sparse groups cannot establish universal rule.

In [6]:
bug = bug_pair_analysis(species)
for key, table in bug.items():
    print(key.upper())
    display(table.round(2))
    table.to_csv(OUTPUT / f"bug_{key}.csv", index=False)
fig, ax = plt.subplots(figsize=(10, 4.5))
comparison = bug["stage"].pivot(index="stage", columns="bug", values="median_residual").reindex(STAGE_ORDER)
comparison = comparison.rename(columns={False: "Other dual types", True: "Dual Bug"})
comparison.plot.bar(ax=ax, color=["#94a3b8", "#b45309"], width=.75)
ax.axhline(0, color="grey", linestyle="--")
ax.set(xlabel="Stage; support counts in table above", ylabel="Median BST minus stage median", title="Dual Bug association changes across stages")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(OUTPUT / "bug_stage_comparison.png", dpi=180)
plt.show()
regular_presence = presence[presence.rarity.eq("regular") & ~presence["sparse"]].copy()
regular_presence["rank_low_to_high"] = regular_presence.groupby("stage").median_residual.rank(method="min")
display(regular_presence[regular_presence.type.eq("bug")].round(2))

STAGE


,stage,bug,count,families,median_bst,median_residual,sparse
0,s1c2,False,71,71,320.0,12.0,False
1,s1c2,True,19,19,304.0,-4.0,False
2,s1c3,False,38,38,291.0,-9.0,False
3,s1c3,True,3,3,260.0,-40.0,True
4,s2c2,False,105,100,490.0,5.0,False
5,s2c2,True,26,23,467.5,-17.5,False
6,s2c3,False,48,48,405.0,0.0,False
7,s2c3,True,5,5,360.0,-45.0,True
8,s3c3,False,72,71,526.5,6.5,False
9,s3c3,True,9,8,411.0,-109.0,True


GENERATION


,stage,generation,bug,count,families,median_bst,median_residual,sparse
0,s1c2,1,False,8,8,330.0,22.0,True
1,s1c2,1,True,3,3,305.0,-3.0,True
2,s1c2,2,False,10,10,330.0,22.0,False
3,s1c2,2,True,3,3,265.0,-43.0,True
4,s1c2,3,False,9,9,300.0,-8.0,True
...,...,...,...,...,...,...,...,...
80,single,7,False,15,15,490.0,12.0,False
81,single,7,True,2,2,570.0,92.0,True
82,single,8,False,7,7,505.0,27.0,True
83,single,9,False,19,19,570.0,92.0,False


PARTNER


,stage,partner,bug,count,families,median_bst,median_residual,sparse
0,s1c2,dark,False,8,8,339.0,31.0,True
1,s1c2,dragon,False,3,3,362.0,54.0,True
2,s1c2,electric,False,4,4,284.5,-23.5,True
3,s1c2,electric,True,1,1,319.0,11.0,True
4,s1c2,fairy,False,2,2,282.5,-25.5,True
...,...,...,...,...,...,...,...,...
133,single,rock,False,10,10,492.5,14.5,False
134,single,rock,True,1,1,505.0,27.0,True
135,single,steel,False,8,8,520.0,42.0,True
136,single,steel,True,1,1,484.0,6.0,True


/var/folders/h8/kn6p4vt50_v066v_y6f9qp9w0000gn/T/ipykernel_16200/1089472866.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,rarity,stage,type,count,families,median_bst,median_residual,sparse,rank_low_to_high
23,regular,s1c2,bug,19,19,304.0,-4.0,False,1.0
58,regular,s2c2,bug,26,23,467.5,-17.5,False,1.0


## Compare identical swapped pairs

Compare A/B versus B/A only within same unordered pair, stage and rarity. Mark either direction below ten observations sparse. Family and generation differences still prevent causal interpretation.

In [7]:
swapped = compare_exact_type_ordering(species)
display(swapped.sort_values("difference").round(2))
swapped.to_csv(OUTPUT / "exact_swapped_pairs.csv", index=False)
print("Pairs with >=10 records in both directions:", int((~swapped["sparse"]).sum()))

,rarity,stage,pair,forward,reverse,n_forward,n_reverse,forward_median,reverse_median,difference,sparse
31,regular,single,dark / ghost,dark/ghost,ghost/dark,1,1,380.0,485.0,-105.0,True
4,regular,s1c2,bug / poison,bug/poison,poison/bug,2,1,277.5,330.0,-52.5,True
17,regular,s2c2,dragon / flying,dragon/flying,flying/dragon,1,1,490.0,535.0,-45.0,True
33,regular,single,dragon / water,dragon/water,water/dragon,1,1,475.0,505.0,-30.0,True
5,regular,s1c2,bug / rock,bug/rock,rock/bug,1,1,325.0,355.0,-30.0,True
9,regular,s1c2,poison / water,poison/water,water/poison,2,1,312.5,335.0,-22.5,True
22,regular,s2c2,poison / water,poison/water,water/poison,1,1,495.0,515.0,-20.0,True
29,regular,single,dark / dragon,dark/dragon,dragon/dark,1,1,570.0,590.0,-20.0,True
6,regular,s1c2,bug / water,bug/water,water/bug,2,1,249.5,269.0,-19.5,True
19,regular,s2c2,ghost / grass,ghost/grass,grass/ghost,2,2,484.0,494.0,-10.0,True


Pairs with >=10 records in both directions: 0


## Bridge to correction features

Original map functions remain available. Full-population maps below are descriptive examples only. Training must fit maps on training folds, with out-of-fold values for training rows and explicit handling of missing support.

In [8]:
display(fit_dual_type_dev_map(groups).head())
mono_map, _, _ = fit_mono_type_dev_map(df)
display(mono_map.head())

,rarity,stage,type,dev_z_t1,dev_z_t2,dev_z_t1_isna,dev_z_t2_isna
7,legendary,single,dragon,2.302015,3.058324,0,0
10,legendary,single,fighting,0.033089,-0.723220,0,0
11,legendary,single,fire,0.033089,0.033089,0,0
14,legendary,single,grass,-0.723220,-0.534143,0,0
19,legendary,single,psychic,0.033089,-0.155989,0,0


,rarity,stage,type_1,mono_dev_z,mono_dev_z_isna
9,legendary,single,electric,0.022014,0
18,legendary,single,normal,2.475003,0
19,legendary,single,psychic,0.294568,0
22,legendary,single,water,0.294568,0
23,regular,s1c2,bug,-1.368013,0


## Interpretation

Keep observed associations, exceptions and support together. Prefer raw categorical types as initial model inputs; promote engineered deviations only after leakage-safe forward comparisons show incremental value.

## Original author notes

Original comments preserved below. They document exploration, not validated rules. Audit sections above qualify or challenge them.

### dist_type_plots

```python
# Here I will just plot how different pokes grouped by types and stages and how their types affect their bst wrt to the baseline of the median of the stage.
# This will just check how the bst of each pokemon deviates wrt the median of their stage for each type (mono, t1 or t2)
# This function will basically plot the following: For each stage and rarity, it will scan for pokes with combinations with the same type as primary and secondary. If more than a min threshold exist, it will then evaluate and plot how type ordering may affect the resulting bst. 
```

### dist_type_tables

```python
# Here I will just check that the underlying functions to get the plots in `notebooks/dist_type_plots` are working and display the information I desire.
```

### Original exploratory notes

Historical comments and docstrings; current analysis above qualifies their claims.

```text
# Change to project root (parent of notebooks/)
# For imports from pokeml/
# Verify: /path/to/project_root
# Load the minimal_pkdx
```